# FinBERT Aspect Classifier — Colab Runner

This notebook clones the repo, installs dependencies, uploads your data, and runs training.  
All heavy logic lives in the `.py` files — this notebook is just the launcher.

**Before running:** Runtime → Change runtime type → **T4 GPU**


## 1 · Clone repo & install

In [11]:
import os

REPO_URL    = "https://github.com/Nas-Mohd/economic-news-sentiment.git"
BRANCH      = "main"
REPO_DIR    = "/content/economic-news-sentiment"

from google.colab import userdata

token = userdata.get("Github_Token")

# Change to a directory that definitely exists
os.chdir('/content')

# Remove existing directory if any (to avoid conflicts)
if os.path.exists(REPO_DIR):
    !rm -rf {REPO_DIR}

# Clone using token for authentication
!git clone https://{token}@github.com/Nas-Mohd/economic-news-sentiment.git {REPO_DIR}

# Optional: configure git user info (for later commits)
!git config --global user.email "anasamohammad.school@gmail.com"
!git config --global user.name "Nas-Mohd"

# Now change into the repo (use absolute path)
os.chdir(REPO_DIR)

# Install dependencies using absolute path
!pip install -r labeling/requirements.txt --quiet

Cloning into '/content/economic-news-sentiment'...
remote: Enumerating objects: 438, done.
remote: Counting objects: 100% (55/55), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 438 (delta 33), reused 40 (delta 21), pack-reused 383 (from 1)
Receiving objects: 100% (438/438), 503.64 KiB | 4.80 MiB/s, done.
Resolving deltas: 100% (276/276), done.


In [12]:
!git -C /content/economic-news-sentiment add .
!git -C /content/economic-news-sentiment commit -m "modifications"
!git -C /content/economic-news-sentiment push

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
Everything up-to-date


## 2 · Mount Google Drive
*Checkpoints and outputs survive disconnection here.*

In [7]:
# No need to upload – the data is already inside the cloned repo
DATA_PATH = "/content/economic-news-sentiment/labeling/data/prob_labels.csv"

# Quick sanity check
import pandas as pd
df = pd.read_csv(DATA_PATH)
print(f"{len(df)} rows  |  columns: {list(df.columns)}")
df.head(3)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Outputs → /content/drive/MyDrive/finbert_outputs


## 3 · Upload your data

In [8]:
from google.colab import files

uploaded  = files.upload()           # select your labels.csv
DATA_PATH = list(uploaded.keys())[0]
print(f"Uploaded: {DATA_PATH}")

# Quick sanity check
import pandas as pd
df = pd.read_csv(DATA_PATH)
print(f"{len(df)} rows  |  columns: {list(df.columns)}")
df.head(3)


IndexError: list index out of range

## 4 · Verify GPU

In [ ]:
!nvidia-smi | head -15
import torch
print(f"\nPyTorch sees CUDA: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")


## 5 · Edit config (optional)

Open `config.py` in the repo and change hyperparameters there.  
The defaults are already set correctly based on your data diagnostics:
- Fiscal `pos_weight` capped at 8
- Abstain zone masked at [0.35, 0.55]  
- 3-phase training with early stopping

Re-run cell 1 (`git pull`) after any edits pushed to GitHub.


## 6 · Train

In [ ]:
# Pass OUTPUT_DIR so checkpoints go to Drive
%cd {REPO_DIR}

!python train.py --data "{DATA_PATH}" --output_dir "{OUTPUT_DIR}"


## 7 · Plot results

In [ ]:
%cd {REPO_DIR}
!python utils/plot.py --output_dir "{OUTPUT_DIR}"

# Display inline
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

for fname in ['training_curves.png', 'comparison_chart.png']:
    path = f"{OUTPUT_DIR}/{fname}"
    import os
    if os.path.exists(path):
        img = mpimg.imread(path)
        plt.figure(figsize=(14, 6))
        plt.imshow(img); plt.axis('off')
        plt.title(fname); plt.show()


## 8 · Download results

In [ ]:
from google.colab import files

for fname in ['comparison_results.csv', 'comparison_chart.png', 'training_curves.png']:
    path = f"{OUTPUT_DIR}/{fname}"
    if os.path.exists(path):
        files.download(path)
        print(f"Downloaded: {fname}")


In [ ]:
import torch
import sys
import os

# Add the repo path to Python path (adjust if needed)
REPO_DIR = "/content/finbert"
sys.path.insert(0, REPO_DIR)

# Your output directory from training (where checkpoints were saved)
OUTPUT_DIR = '/content/drive/MyDrive/finbert_outputs'   # change if different

# Find the best model checkpoint
import glob
checkpoint_files = glob.glob(f"{OUTPUT_DIR}/*.pth")
if not checkpoint_files:
    raise FileNotFoundError(f"No .pth checkpoint found in {OUTPUT_DIR}")
# Prefer best_model.pth or the one with highest epoch F1
best_model_path = None
for f in checkpoint_files:
    if 'baseline' in f:
        best_model_path = f
        break
if not best_model_path:
    best_model_path = checkpoint_files[0]   # take first
print(f"Loading checkpoint: {best_model_path}")

# Load model architecture (your custom classes)
from models.model import BaselineClassifier, AspectClassifier
from utils.config import MODEL_CFG, TRAIN_CFG   # contains model name etc.

# Determine which model was used – check checkpoint's model_name
# Default: Baseline (ProsusAI/finbert). You can change to 'modern' if you trained that.
# For simplicity, we'll try both; but you know which one you trained.
# If you trained the fine-tuned model (Modern-FinBERT-large), use AspectClassifier.
# If you trained baseline only, use BaselineClassifier.
# Here we assume you want the fine-tuned model (the one that ran Phase 1+2).
model = BaselineClassifier(MODEL_CFG)   # uses MODEL_CFG.model_name = "beethogedeon/Modern-FinBERT-large"

# Load state dict (removing potential 'module.' prefix if saved from DataParallel)
state = torch.load(best_model_path, map_location='cpu')
if 'model_state_dict' in state:
    state = state['model_state_dict']
# Remove 'module.' prefix if present
new_state = {k.replace('module.', ''): v for k, v in state.items()}
model.load_state_dict(new_state, strict=False)   # strict=False in case head keys differ slightly

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
model.eval()

# Load tokenizer (use same as in config)
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('ProsiusAI/finbert')
import json

# Load the training log (choose baseline or finetune)
log_path = f"{OUTPUT_DIR}/baseline_training_log.json"   # or 'finetune_training_log.json'
if os.path.exists(log_path):
    with open(log_path, 'r') as f:
        log = json.load(f)
    # The thresholds are likely stored in log['best_thresholds'] or similar
    # Let's search for thresholds
    thresholds = None
    if 'thresholds' in log:
        thresholds = log['thresholds']
    elif 'best_thresholds' in log:
        thresholds = log['best_thresholds']
    elif 'tuned_thresholds' in log:
        thresholds = log['tuned_thresholds']
    if thresholds:
        print(f"Loaded tuned thresholds: {thresholds}")
    else:
        thresholds = [0.5] * 6   # fallback
else:
    thresholds = [0.5] * 6
    print("No thresholds file found. Using default 0.5.")
# Aspect columns (order should match training)
aspect_cols = ['Monetary_Financial', 'Inflation_Prices', 'Real_Economic_Activity',
               'Labor_Consumption', 'Fiscal_Government', 'External_Sector']

def predict_sentence(text, thresholds=[0.5]*6):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=128, padding=True)
    vocab_size = tokenizer.vocab_size
    if inputs['input_ids'].max() >= vocab_size:
        inputs['input_ids'] = torch.clamp(inputs['input_ids'], 0, vocab_size - 1)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = model(inputs['input_ids'], inputs['attention_mask'])
        probs = torch.sigmoid(logits).cpu().numpy()[0]
    binary = (probs >= thresholds).astype(int)
    return probs, binary

# Interactive loop
print("\n" + "="*60)
print("Aspect Classifier Ready")
print("Aspects: " + ", ".join(aspect_cols))
print("Enter a sentence (or 'quit' to exit):")
print("="*60)

while True:
    sentence = input("\n> ").strip()
    if sentence.lower() in ('quit', 'exit', 'q'):
        break
    if not sentence:
        continue
    probs, binary = predict_sentence(sentence)
    print("\nProbabilities:")
    for asp, p in zip(aspect_cols, probs):
        print(f"  {asp:25s} : {p:.4f}")
    print("\nBinary (threshold=0.5):")
    for asp, b in zip(aspect_cols, binary):
        print(f"  {asp:25s} : {b}")